In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
df = pd.read_csv('diabetes_pima.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [4]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)
X.shape

(768, 8)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [7]:
import tensorflow
from tensorflow import keras
from keras import Sequential, Input
from keras.layers import Dense, BatchNormalization

model = Sequential(
    [
        Input(shape=(X_train.shape[1],)),
        Dense(32, activation='relu'),
        BatchNormalization(),
        Dense(16, activation='relu'),
        BatchNormalization(),
        Dense(1, activation='sigmoid')
    ]
)
model.summary()

/Users/othmaneabderrazik/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,025 (4.00 KB)

 Trainable params: 929 (3.63 KB)

 Non-trainable params: 96 (384.00 B)

In [8]:
model.compile(
    optimizer='SGD',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32
)

Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5385 - loss: 0.8323 - val_accuracy: 0.6016 - val_loss: 0.6805
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5958 - loss: 0.7386 - val_accuracy: 0.6341 - val_loss: 0.6506
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6436 - loss: 0.6649 - val_accuracy: 0.6341 - val_loss: 0.6284
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6229 - loss: 0.6465 - val_accuracy: 0.6585 - val_loss: 0.6077
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6926 - loss: 0.5910 - val_accuracy: 0.6911 - val_loss: 0.5891
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7108 - loss: 0.5653 - val_accuracy: 0.7317 - val_loss: 0.5731
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7095 - loss: 0.5915 - val_accuracy: 0.7317 - val_loss: 0.5583
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7137 - loss: 0.5549 - val_accuracy: 0.7317 - v

In [9]:
import keras_tuner as kt
from keras.layers import BatchNormalization

In [10]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],))) 

    model.add(Dense(
        units=hp.Int('units_layer_1', min_value=16, max_value=128, step=16),
        activation='relu'
    ))

    model.add(BatchNormalization())

    model.add(Dense(
        units=hp.Int('units_layer_2', min_value=16, max_value=128, step=16),
        activation='relu'
    ))

    model.add(BatchNormalization())

    model.add(Dense(1, activation='sigmoid'))

    optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop'])

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [11]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,
    directory='trials_dir',
    project_name='diabetes_tuning'
)

In [12]:
tuner.search(
    X_train, y_train,
    validation_split=0.2,
    epochs=50
)

Trial 3 Complete [00h 00m 02s]
val_accuracy: 0.8292682766914368

Best val_accuracy So Far: 0.8292682766914368
Total elapsed time: 00h 00m 06s


In [13]:
model = tuner.get_best_models(num_models=1)[0]
model.summary()

/Users/othmaneabderrazik/Library/Python/3.9/lib/python/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 12 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 48)             │           432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48)             │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,489 (5.82 KB)

 Trainable params: 1,361 (5.32 KB)

 Non-trainable params: 128 (512.00 B)

In [14]:
model.fit(
    X_train, y_train,
    validation_split=0.2,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32
)

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8293 - loss: 0.3655 - val_accuracy: 0.7468 - val_loss: 0.5228
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8318 - loss: 0.3569 - val_accuracy: 0.7403 - val_loss: 0.5289
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8817 - loss: 0.3233 - val_accuracy: 0.7273 - val_loss: 0.5282
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8373 - loss: 0.3485 - val_accuracy: 0.7208 - val_loss: 0.5418
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8380 - loss: 0.3416 - val_accuracy: 0.7338 - val_loss: 0.5412
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 988us/step - accuracy: 0.8596 - loss: 0.3356 - val_accuracy: 0.7078 - val_loss: 0.5454
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 963us/step - accuracy: 0.8713 - loss: 0.3138 - val_accuracy: 0.6818 - val_loss: 0.5509
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 976us/step - accuracy: 0.8476 - loss: 0.3591 - val_accuracy: 0.7078 - val

In [15]:
import joblib

model.save('diabetes_model.keras')
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

In [16]:
from keras.models import load_model

model = load_model('diabetes_model.keras')
scaler = joblib.load('scaler.pkl')